<a href="https://colab.research.google.com/github/mannangrover/Diabities_prediction_system/blob/main/WearableDiabaties.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Update and install deps

In [ ]:
!pip install kagglehub




Downloading Dataset

In [ ]:

import kagglehub
from kagglehub import KaggleDatasetAdapter
from google.colab import userdata
import os
import shutil  # Library for moving files

# Authentication
os.environ['KAGGLE_USERNAME'] = userdata.get('kaggle_username')
os.environ['KAGGLE_KEY'] = userdata.get('kaggle_token')

# Downloading dataset
print("Downloading dataset...")
dataset_path = kagglehub.dataset_download("dariushbahrami/cdc-brfss-survey-2021")
print(f"Cache location: {dataset_path}")

# Moving File to Home/content
real_filename = "LLCP2021.csv"
source_path = os.path.join(dataset_path, real_filename)
destination_path = f"/content/{real_filename}"

if os.path.exists(source_path):
    shutil.copy(source_path, destination_path)
    print(f"\nSUCCESS: File copied to your home folder: {destination_path}")

    # Loading Dataset
    print("Loading into Hugging Face Dataset...")
    hf_dataset = kagglehub.load_dataset(
      KaggleDatasetAdapter.HUGGING_FACE,
      "dariushbahrami/cdc-brfss-survey-2021",
      real_filename,
    )
    print("Dataset loaded successfully!")
    print(hf_dataset)

else:
    print(f"\nError: Could not find '{real_filename}' in the downloaded folder.")
    print("Files found:", os.listdir(dataset_path))

100%|██████████| 48.2M/48.2M [00:02<00:00, 22.7MB/s]

Extracting files...


Cache location: /root/.cache/kagglehub/datasets/dariushbahrami/cdc-brfss-survey-2021/versions/1

SUCCESS: File copied to your home folder: /content/LLCP2021.csv
Loading into Hugging Face Dataset...


/tmp/ipykernel_1212/3284363369.py:27: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  hf_dataset = kagglehub.load_dataset(


Using Colab cache for faster access to the 'cdc-brfss-survey-2021' dataset.
Dataset loaded successfully!
Dataset({
    features: ['_STATE', 'FMONTH', 'IDATE', 'IMONTH', 'IDAY', 'IYEAR', 'DISPCODE', 'SEQNO', '_PSU', 'CTELENM1', 'PVTRESD1', 'COLGHOUS', 'STATERE1', 'CELPHON1', 'LADULT1', 'COLGSEX', 'NUMADULT', 'LANDSEX', 'NUMMEN', 'NUMWOMEN', 'RESPSLCT', 'SAFETIME', 'CTELNUM1', 'CELLFON5', 'CADULT1', 'CELLSEX', 'PVTRESD3', 'CCLGHOUS', 'CSTATE1', 'LANDLINE', 'HHADULT', 'SEXVAR', 'GENHLTH', 'PHYSHLTH', 'MENTHLTH', 'POORHLTH', 'PRIMINSR', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', 'EXERANY2', 'BPHIGH6', 'BPMEDS', 'CHOLCHK3', 'TOLDHI3', 'CHOLMED3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'ASTHMA3', 'ASTHNOW', 'CHCSCNCR', 'CHCOCNCR', 'CHCCOPD3', 'ADDEPEV3', 'CHCKDNY2', 'DIABETE4', 'DIABAGE3', 'HAVARTH5', 'ARTHEXER', 'ARTHEDU', 'LMTJOIN3', 'ARTHDIS2', 'JOINPAI2', 'MARITAL', 'EDUCA', 'RENTHOM1', 'NUMHHOL3', 'NUMPHON3', 'CPDEMO1B', 'VETERAN3', 'EMPLOY1', 'CHILDREN', 'INCOME3', 'PREGNANT', 'WEIGHT2', 'HEIGHT

Cleaning the data and sorting a few metric that we might need

In [ ]:
import pandas as pd
import numpy as np

# Load the Dataset

file_path = '/content/LLCP2021.csv'
print(f"Initiating data load from: {file_path}")
try:
    df = pd.read_csv(file_path)
    print(f"Data loaded successfully. Initial Dimensions: {df.shape}")
except FileNotFoundError:
    print("Error: File not found. Please upload LLCP2021.csv.")
    raise

# Feature Subset

selected_features = [
    'DIABETE4',   # Target Variable
    '_BMI5',      # Body Mass Index
    'EXERANY2',   # Exercise Status (Last 30 Days)
    '_TOTINDA',   # Physical Activity Index
    'DIFFWALK',   # Difficulty Walking
    'GENHLTH',    # General Health Rating
    '_PHYS14D',   # Physical Health (Days not good)
    'BPMEDS',     # Blood Pressure Medication Use
    'BPHIGH6',    # History of High Blood Pressure
    'TOLDHI3',    # History of High Cholesterol
    '_MICHD',     # History of Coronary Heart Disease/MI
    'SMOKE100',   # Lifetime Smoking Status
    'ALCDAY5',    # Alcohol Consumption Frequency
    '_FRUTSU1',   # Daily Fruit Consumption
    '_VEGESU1',   # Daily Vegetable Consumption
    '_AGEG5YR',   # Age Category
    'SEXVAR'      # Biological Sex
]


df_clean = df[selected_features].copy()

# Target Variable Standardization (DIABETE4)

# 0 = No Diabetes (Codes 2, 3)
# 1 = Diabetes or Pre-diabetes (Codes 1, 4)


df_clean = df_clean[~df_clean['DIABETE4'].isin([7, 9])]
df_clean['DIABETE4'] = df_clean['DIABETE4'].replace({
    2: 0,
    3: 0,
    1: 1,
    4: 1
})

# Feature Engineering and Cleaning

# BMI5 (Body Mass Index)
# Code 9999 denotes missing.
df_clean['_BMI5'] = df_clean['_BMI5'].replace(9999, np.nan) / 100

# ALCDAY5 (Alcohol Frequency)
# Code 888 = 0 drinks.
# Codes 101-107 = Days per week.
# Codes 201-230 = Days per month.
# Codes 777/999 = Missing.
def clean_alcohol_data(value):
    if value == 888:
        return 0
    elif 101 <= value <= 107:
        return (value - 100) * 4.3  #  weekly to monthly
    elif 201 <= value <= 230:
        return value - 200          # Already monthly frequency
    else:
        return np.nan               # Treat refusal/don't know as missing

df_clean['ALCDAY5'] = df_clean['ALCDAY5'].apply(clean_alcohol_data)

#  _FRUTSU1 & _VEGESU1 (Dietary Intake)
# Values contain 2 implied decimal places. Code 9999 denotes missing.
df_clean['_FRUTSU1'] = df_clean['_FRUTSU1'].replace(9999, np.nan) / 100
df_clean['_VEGESU1'] = df_clean['_VEGESU1'].replace(9999, np.nan) / 100

# _TOTINDA (Physical Activity Index)
# 1 = Active, 2 = Inactive converted to 0
df_clean['_TOTINDA'] = df_clean['_TOTINDA'].replace({2: 0, 9: np.nan})

# GENHLTH (General Health)
# Scale 1-5. Removed 7/9 (Refused/Missing).
df_clean['GENHLTH'] = df_clean['GENHLTH'].replace([7, 9], np.nan)

# _PHYS14D (Physical Health Status)
# Removed 9 (Missing).
df_clean['_PHYS14D'] = df_clean['_PHYS14D'].replace(9, np.nan)

# Binary Variable Normalization-
# For columns where 1=Yes, 2=No converted
binary_cols = ['EXERANY2', 'DIFFWALK', 'SMOKE100', '_MICHD', 'BPMEDS', 'BPHIGH6', 'TOLDHI3']

# Note regarding TOLDHI3: Dataset sometimes uses 1=Yes, 2=No coverted to 0
for col in binary_cols:
    df_clean[col] = df_clean[col].replace({2: 0, 7: np.nan, 9: np.nan})

# Demographics
# _AGEG5YR: 14 denotes missing/refused.
df_clean['_AGEG5YR'] = df_clean['_AGEG5YR'].replace(14, np.nan)

# Missing Value Handling
# Drop any row containing NaN values to ensure dataset integrity.
df_clean = df_clean.dropna()
print(f"Dimensions after cleaning missing values: {df_clean.shape}")

# Class Balancing (Undersampling)
# To prevent model bias towards the majority class (Non-Diabetic),
# we undersample the negative class to match the positive class count.

diabetic_records = df_clean[df_clean['DIABETE4'] == 1]
non_diabetic_records = df_clean[df_clean['DIABETE4'] == 0]

# non-diabetic records
non_diabetic_downsampled = non_diabetic_records.sample(n=len(diabetic_records), random_state=42)

# Concatenate and shuffle
df_balanced = pd.concat([diabetic_records, non_diabetic_downsampled])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final Balanced Dataset Dimensions: {df_balanced.shape}")
print(f"Class Distribution (DIABETE4):\n{df_balanced['DIABETE4'].value_counts()}")

# 7. File Export
output_filename = '/content/diabetes_binary_5050split_wearable_prototype.csv'
df_balanced.to_csv(output_filename, index=False)
print(f"Processing complete. Data exported to: {output_filename}")

Initiating data load from: /content/LLCP2021.csv
Data loaded successfully. Initial Dimensions: (438693, 303)
Dimensions after cleaning missing values: (119485, 17)
Final Balanced Dataset Dimensions: (67234, 17)
Class Distribution (DIABETE4):
DIABETE4
1.0    33617
0.0    33617
Name: count, dtype: int64
Processing complete. Data exported to: /content/diabetes_binary_5050split_wearable_prototype.csv


### **Target Variable (The Output)**

* **`DIABETE4`**
*  Does the user have Diabetes or Pre-diabetes?
* **Logic:** `0` = Healthy, `1` = Diabetes or Pre-diabetes. (Refusals removed).



---

### **Wearable & Physical Signals (The "Watch" Data)**

* **`_BMI5`**
*  Body Mass Index (calculated from height/weight).
* **Formula:** `Original Value / 100` (e.g., 2500 becomes 25.0).


* **`EXERANY2`**
* Did they do *any* exercise in the last 30 days?
* **Logic:** `1` = Yes, `0` = No.


* **`_TOTINDA`**
*  Physical Activity Index (Are they "Active" vs. "Sedentary"?).
* **Logic:** `1` = Active, `0` = Inactive.


* **`DIFFWALK`**
*  Do they have serious difficulty walking or climbing stairs?
* **Logic:** `1` = Yes, `0` = No.


* **`_PHYS14D`**
*  How many days was their physical health "not good" this month?
* **Logic:** `1` = Zero days (Good), `2` = 1-13 days (Okay), `3` = 14+ days (Bad).



---

### **Medical History (User Profile)**

* **`BPMEDS`**
*  Are they currently taking Blood Pressure medication?
* **Logic:** `1` = Yes, `0` = No.


* **`BPHIGH6`**
*  Have they ever been told they have High Blood Pressure?
* **Logic:** `1` = Yes, `0` = No.


* **`TOLDHI3`**
*  Have they ever been told they have High Cholesterol?
* **Logic:** `1` = Yes, `0` = No.


* **`_MICHD`**
*  Heart Disease History (Heart Attack or Angina).
* **Logic:** `1` = Yes, `0` = No.


* **`GENHLTH`**
*  User's rating of their own health (1=Excellent to 5=Poor).
* **Logic:** Kept as 1-5 scale (Removed 7/9 refusals).



---

### **Habits & Demographics**

* **`_FRUTSU1`**
*  Fruit "Diet Score" (Times per day).
* **Formula:** `Original Value / 100` (e.g., 200 = 2.0 times/day).


* **`_VEGESU1`**
*  Vegetable "Diet Score" (Times per day).
* **Formula:** `Original Value / 100`.


* **`ALCDAY5`**
*  Alcohol frequency (Drinks per Month).
* **Formula:** Weekly values multiplied by `4.3` to get monthly; `888` (None) set to `0`.


* **`SMOKE100`**
*  Lifetime Smoker? (Smoked >100 cigs in entire life).
* **Logic:** `1` = Yes, `0` = No.


* **`_AGEG5YR`**
*  Age Bracket (1 = Age 18-24 ... 13 = Age 80+).
* **Logic:** 5-year increments.


* **`SEXVAR`**
*  Biological Sex.
* **Logic:** `1` = Male, `2` = Female.

Training using XGBoost with GPU runtime

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Data Loading
INPUT_FILE = '/content/diabetes_binary_5050split_wearable_prototype.csv'
MODEL_FILE = '/content/diabetes_wearable_prototype.json'

print(f"Loading dataset from {INPUT_FILE}...")
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"loaded Shape: {df.shape}")
except FileNotFoundError:
    print(f"ERROR: File {INPUT_FILE} not found. clean first")
    raise

# Preprocessing
# Features (X) and Target (y)
target_col = 'DIABETE4'
X = df.drop(columns=[target_col])
y = df[target_col]

# 80% Train, 20% Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training Data: {X_train.shape}")
print(f"Testing Data:  {X_test.shape}")

# Model Initialization (GPU Accelerated)
# We use the XGBClassifier with 'gpu_hist' tree method.
# This offloads the heavy decision tree construction to the GPU.

print("\nXGBoost starting...")

model = xgb.XGBClassifier(
    # Core GPU Parameters
    device='cuda',            # CUDA usage
    tree_method='hist',       # Use histogram-based algorithm (required for efficient GPU usage)

    # Model Hyperparameters (Optimized for prototype)
    n_estimators=500,         # Number of trees
    learning_rate=0.05,       # Step size shrinkage
    max_depth=6,              # Maximum depth of a tree
    subsample=0.8,            # Subsample ratio of the training instances
    colsample_bytree=0.8,     # Subsample ratio of columns when constructing each tree

    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

# Training

print("Starting training process...")
start_time = time.time()

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

end_time = time.time()
print(f"Training complete in {end_time - start_time:.2f} seconds.")

# Evaluation
print("\n--- Model Performance Metrics ---")

# Generate predictions
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nClassification Report:")
print(class_report)

print("Confusion Matrix:")
print(conf_matrix)

# Feature Importance Analysis
importance = model.feature_importances_
feature_names = X.columns
feat_imp = pd.DataFrame({'Feature': feature_names, 'Importance': importance})
feat_imp = feat_imp.sort_values(by='Importance', ascending=False)

print("\n--- 10 Most Critical Features ---")
print(feat_imp.head(10))

#  Model Export
model.save_model(MODEL_FILE)
print(f"\nModel saved to: {MODEL_FILE}")
print("load model using model.load_model()")

Loading dataset from /content/diabetes_binary_5050split_wearable_prototype.csv...
loaded Shape: (67234, 17)
Training Data: (53787, 16)
Testing Data:  (13447, 16)

XGBoost starting...
Starting training process...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:15:56] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Training complete in 1.76 seconds.

--- Model Performance Metrics ---
Accuracy: 0.6723

Classification Report:
              precision    recall  f1-score   support

         0.0       0.68      0.65      0.66      6724
         1.0       0.66      0.70      0.68      6723

    accuracy                           0.67     13447
   macro avg       0.67      0.67      0.67     13447
weighted avg       0.67      0.67      0.67     13447

Confusion Matrix:
[[4340 2384]
 [2023 4700]]

--- 10 Most Critical Features ---
     Feature  Importance
4    GENHLTH    0.200488
6     BPMEDS    0.128774
8    TOLDHI3    0.126178
11   ALCDAY5    0.079259
3   DIFFWALK    0.076470
9     _MICHD    0.055982
0      _BMI5    0.051682
14  _AGEG5YR    0.048531
15    SEXVAR    0.044694
2   _TOTINDA    0.040207

Model saved to: /content/diabetes_wearable_prototype.json
load model using model.load_model()


/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [21:15:58] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Tuning using multiple techniques bit bang





In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, randint

# Define the parameter grid
param_dist = {
    'learning_rate': uniform(0.01, 0.2),    # Step size
    'max_depth': randint(3, 10),            # Tree depth (prevent overfitting)
    'n_estimators': randint(100, 1000),     # Number of trees
    'subsample': uniform(0.6, 0.4),         # % of data used per tree
    'colsample_bytree': uniform(0.6, 0.4),  # % of features used per tree
    'gamma': uniform(0, 0.5)                # Min loss reduction required
}

# Setup the Search
clf = xgb.XGBClassifier(
    device='cuda',
    tree_method='hist',
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

random_search = RandomizedSearchCV(
    clf,
    param_distributions=param_dist,
    n_iter=25,              # Try 25 different combinations
    scoring='recall',       # Optimize for RECALL (Catching more diabetics)
    cv=3,                   # 3-fold cross-validation
    verbose=1,
    n_jobs=1                # XGBoost handles parallelism internally
)

print("Starting Hyperparameter Tuning (Optimizing for Recall)...")
random_search.fit(X_train, y_train)

print(f"\nBest Recall Score: {random_search.best_score_:.4f}")
print("Best Parameters Found:")
print(random_search.best_params_)

# Test the Best Model
best_model = random_search.best_estimator_
y_pred_optimized = best_model.predict(X_test)

print("\n--- Optimized Classification Report ---")
print(classification_report(y_test, y_pred_optimized))

Starting Hyperparameter Tuning (Optimizing for Recall)...
Fitting 3 folds for each of 25 candidates, totalling 75 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:15:58] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:15:59] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:16:01] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:16:03] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/xgboost/training.py:


Best Recall Score: 0.7072
Best Parameters Found:
{'colsample_bytree': np.float64(0.7050667768305788), 'gamma': np.float64(0.36416037358867637), 'learning_rate': np.float64(0.17774152919169509), 'max_depth': 4, 'n_estimators': 129, 'subsample': np.float64(0.8690870491229928)}

--- Optimized Classification Report ---
              precision    recall  f1-score   support

         0.0       0.69      0.65      0.67      6724
         1.0       0.67      0.71      0.69      6723

    accuracy                           0.68     13447
   macro avg       0.68      0.68      0.68     13447
weighted avg       0.68      0.68      0.68     13447



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [21:17:23] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Found the best parameters using hyper tuning bit banging. Training new model based on found best parameters

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

#  Configuration
INPUT_FILE = '/content/diabetes_binary_5050split_wearable_prototype.csv'
MODEL_FILE = '/content/diabetes_wearable_final.json'

# Load Data
df = pd.read_csv(INPUT_FILE)
X = df.drop(columns=['DIABETE4'])
y = df['DIABETE4']

#  Apply Optimal Parameters
best_params = {
    'colsample_bytree': 0.8080409037504724,
    'gamma': 0.2168148501045089,
    'learning_rate': 0.05533870666155276,
    'max_depth': 4,
    'n_estimators': 478,
    'subsample': 0.7592918880823795,

    # System parameters
    'device': 'cuda',
    'tree_method': 'hist',
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'random_state': 42
}

print(f"Training Final Model with {best_params['n_estimators']} trees...")

# Train
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X_train, y_train)

#  Final Verification
y_pred = final_model.predict(X_test)
print("\n--- Final Production Metrics ---")
print(classification_report(y_test, y_pred))

# Save Artifacts
final_model.save_model(MODEL_FILE)
print(f"Model saved to: {MODEL_FILE}")

with open('/content/feature_names.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)
print("Feature order saved to: /content/feature_names.pkl")

Training Final Model with 478 trees...

--- Final Production Metrics ---
              precision    recall  f1-score   support

         0.0       0.69      0.65      0.67      6724
         1.0       0.67      0.71      0.69      6723

    accuracy                           0.68     13447
   macro avg       0.68      0.68      0.68     13447
weighted avg       0.68      0.68      0.68     13447

Model saved to: /content/diabetes_wearable_final.json
Feature order saved to: /content/feature_names.pkl


In [ ]:
# ==========================================
# IMPROVED FEATURE ENGINEERING (FIXED)
# Compatible with LLCP2021.csv
# ==========================================

import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer

# Load dataset
df = pd.read_csv('/content/LLCP2021.csv')

print("Original shape:", df.shape)

# ==========================================
# SELECT FEATURES (ONLY AVAILABLE ONES)
# ==========================================
selected_features = [
    'DIABETE4',

    # Physiologic & metabolic
    '_BMI5','_BMI5CAT','BPHIGH6','TOLDHI3','BPMEDS','_MICHD',

    # Health status & recovery indicators
    'GENHLTH','PHYSHLTH','MENTHLTH','POORHLTH',
    '_PHYS14D','_MENT14D',

    # Activity & mobility
    'EXERANY2','_TOTINDA','DIFFWALK',

    # Lifestyle risk
    'SMOKE100','_SMOKER3','ALCDAY5','ADDEPEV3',

    # Diet quality
    '_FRUTSU1','_VEGESU1','FRUTDA2_','VEGEDA2_',

    # Demographics
    '_AGEG5YR','SEXVAR'
]

# Keep only columns that exist (prevents errors)
selected_features = [col for col in selected_features if col in df.columns]

df_new = df[selected_features].copy()

# ==========================================
# TARGET CLEANING
# ==========================================
df_new = df_new[~df_new['DIABETE4'].isin([7, 9])]
df_new['DIABETE4'] = df_new['DIABETE4'].replace({
    2: 0,
    3: 0,
    1: 1,
    4: 1
})

# ==========================================
# FEATURE CLEANING
# ==========================================

# BMI
df_new['_BMI5'] = df_new['_BMI5'].replace(9999, np.nan) / 100
df_new['_BMI5CAT'] = df_new['_BMI5CAT'].replace(9, np.nan)

# Unhealthy days & mental health
for col in ['PHYSHLTH','MENTHLTH','POORHLTH','_PHYS14D','_MENT14D']:
    if col in df_new.columns:
        df_new[col] = df_new[col].replace({77: np.nan, 99: np.nan})

# Binary health indicators
binary_cols = [
    'EXERANY2','DIFFWALK','SMOKE100','_MICHD',
    'BPMEDS','BPHIGH6','TOLDHI3','ADDEPEV3'
]

for col in binary_cols:
    if col in df_new.columns:
        df_new[col] = df_new[col].replace({2: 0, 7: np.nan, 9: np.nan})

# Smoking intensity
if '_SMOKER3' in df_new.columns:
    df_new['_SMOKER3'] = df_new['_SMOKER3'].replace(9, np.nan)

# Alcohol cleaning
def clean_alcohol(value):
    if value == 888:
        return 0
    elif 101 <= value <= 107:
        return (value - 100) * 4.3
    elif 201 <= value <= 230:
        return value - 200
    return np.nan

if 'ALCDAY5' in df_new.columns:
    df_new['ALCDAY5'] = df_new['ALCDAY5'].apply(clean_alcohol)

# Diet conversion
for col in ['_FRUTSU1','_VEGESU1','FRUTDA2_','VEGEDA2_']:
    if col in df_new.columns:
        df_new[col] = df_new[col].replace(9999, np.nan) / 100

# Activity index
if '_TOTINDA' in df_new.columns:
    df_new['_TOTINDA'] = df_new['_TOTINDA'].replace({2: 0, 9: np.nan})

# General health
if 'GENHLTH' in df_new.columns:
    df_new['GENHLTH'] = df_new['GENHLTH'].replace([7, 9], np.nan)

# Age group
if '_AGEG5YR' in df_new.columns:
    df_new['_AGEG5YR'] = df_new['_AGEG5YR'].replace(14, np.nan)

# ==========================================
# HANDLE MISSING VALUES (KEEP DATA)
# ==========================================
imputer = SimpleImputer(strategy='median')
df_new[:] = imputer.fit_transform(df_new)

print("After cleaning:", df_new.shape)

# ==========================================
# SAVE IMPROVED DATASET
# ==========================================
output_file = '/content/diabetes_improved_features.csv'
df_new.to_csv(output_file, index=False)

print("\n✅ Improved dataset saved:")
print(output_file)

Original shape: (438693, 303)
After cleaning: (437711, 26)

✅ Improved dataset saved:
/content/diabetes_improved_features.csv


In [ ]:
# ==========================================
# XGBOOST TRAINING (IMPROVED DATASET)
# ==========================================

import pandas as pd
import numpy as np
import xgboost as xgb
import pickle
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

# ==========================================
# 1️⃣ LOAD IMPROVED DATASET
# ==========================================
INPUT_FILE = '/content/diabetes_improved_features.csv'

df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)

# Separate features & target
X = df.drop(columns=['DIABETE4'])
y = df['DIABETE4']

# ==========================================
# 2️⃣ HANDLE CLASS IMBALANCE (BETTER METHOD)
# ==========================================
# ratio for positive class weighting
scale_pos_weight = (len(y) - sum(y)) / sum(y)
print("scale_pos_weight:", scale_pos_weight)

# ==========================================
# 3️⃣ TRAIN-TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

# ==========================================
# 4️⃣ INITIALIZE XGBOOST (OPTIMIZED)
# ==========================================
model = xgb.XGBClassifier(
    device='cuda',              # remove if GPU unavailable
    tree_method='hist',

    n_estimators=800,
    learning_rate=0.03,
    max_depth=5,

    subsample=0.8,
    colsample_bytree=0.8,

    gamma=0.1,
    min_child_weight=3,

    scale_pos_weight=scale_pos_weight,

    objective='binary:logistic',
    eval_metric='logloss',
    random_state=42
)

# ==========================================
# 5️⃣ TRAIN WITH EARLY STOPPING
# ==========================================
print("\nTraining XGBoost...")

model.fit(
    X_train,
    y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print("Training complete")

# ==========================================
# 6️⃣ PREDICTIONS
# ==========================================
y_prob = model.predict_proba(X_test)[:,1]

# Try better threshold
threshold = 0.70
y_pred = (y_prob > threshold).astype(int)

# ==========================================
# 7️⃣ EVALUATION
# ==========================================
print("\n====== MODEL PERFORMANCE ======\n")

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# ==========================================
# 8️⃣ FEATURE IMPORTANCE
# ==========================================
importances = model.feature_importances_
features = X.columns

feat_imp = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 10 Important Features:")
print(feat_imp.head(10))

# ==========================================
# 9️⃣ SAVE MODEL FOR DEPLOYMENT
# ==========================================
model.save_model('/content/xgboost_diabetes_improved.json')

with open('/content/feature_order_improved.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print("\n✅ Model saved successfully!")


Dataset shape: (437711, 26)
scale_pos_weight: 5.479040232096865
Training shape: (350168, 25)
Testing shape: (87543, 25)

Training XGBoost...
Training complete

====== MODEL PERFORMANCE ======

Accuracy: 0.8241778326079755
ROC-AUC: 0.8244803220746043

Classification Report:
              precision    recall  f1-score   support

         0.0       0.91      0.88      0.89     74031
         1.0       0.44      0.52      0.48     13512

    accuracy                           0.82     87543
   macro avg       0.67      0.70      0.69     87543
weighted avg       0.84      0.82      0.83     87543


Confusion Matrix:
[[65149  8882]
 [ 6510  7002]]

Top 10 Important Features:
     Feature  Importance
2    BPHIGH6    0.387949
6    GENHLTH    0.121986
3    TOLDHI3    0.092553
23  _AGEG5YR    0.059417
14  DIFFWALK    0.055437
4     BPMEDS    0.050663
1   _BMI5CAT    0.037732
5     _MICHD    0.036366
17   ALCDAY5    0.025162
0      _BMI5    0.021144

✅ Model saved successfully!


In [ ]:
# ==========================================
# FINAL XGBOOST HYPERPARAMETER TUNING (IMPROVED)
# ==========================================

import pandas as pd
import numpy as np
import xgboost as xgb

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

# ==========================================
# 1️⃣ LOAD DATA
# ==========================================
df = pd.read_csv('/content/diabetes_improved_features.csv')

X = df.drop(columns=['DIABETE4'])
y = df['DIABETE4']

print("Dataset shape:", df.shape)

# ==========================================
# 2️⃣ HANDLE CLASS IMBALANCE
# ==========================================
scale_pos_weight = (len(y) - sum(y)) / sum(y)
print("scale_pos_weight:", scale_pos_weight)

# ==========================================
# 3️⃣ TRAIN TEST SPLIT
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

# ==========================================
# 4️⃣ BASE MODEL (STRONGER)
# ==========================================
xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',        # 🔥 changed
    tree_method='hist',
    device='cuda',
    scale_pos_weight=scale_pos_weight,
    random_state=42
)

# ==========================================
# 5️⃣ IMPROVED PARAM SEARCH
# ==========================================
param_dist = {
    'n_estimators': [500, 700, 900, 1200],
    'learning_rate': [0.01, 0.02, 0.03],
    'max_depth': [4, 5, 6],
    'min_child_weight': [1, 3, 5],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'gamma': [0, 0.1, 0.3],
    'reg_alpha': [0, 0.1, 1],        # 🔥 added
    'reg_lambda': [1, 1.5, 2]        # 🔥 added
}

# ==========================================
# 6️⃣ RANDOM SEARCH
# ==========================================
random_search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_dist,
    n_iter=30,                 # 🔥 increased
    scoring='roc_auc',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

print("\nRunning Hyperparameter Tuning...\n")
random_search.fit(X_train, y_train)

print("\n⭐ BEST PARAMETERS:")
print(random_search.best_params_)

# ==========================================
# 7️⃣ BEST MODEL
# ==========================================
best_model = random_search.best_estimator_

# ==========================================
# 8️⃣ THRESHOLD OPTIMIZATION (MOST IMPORTANT)
# ==========================================
y_prob = best_model.predict_proba(X_test)[:,1]

print("\n===== THRESHOLD TUNING =====")

best_acc = 0
best_threshold = 0

for t in np.arange(0.6, 0.85, 0.05):
    y_pred = (y_prob > t).astype(int)
    acc = accuracy_score(y_test, y_pred)

    print(f"Threshold {t:.2f} -> Accuracy: {acc:.4f}")

    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"\n🔥 BEST THRESHOLD: {best_threshold}")
print(f"🔥 BEST ACCURACY: {best_acc}")

# ==========================================
# 9️⃣ FINAL EVALUATION
# ==========================================
y_pred_final = (y_prob > best_threshold).astype(int)

print("\n====== FINAL MODEL PERFORMANCE ======\n")

print("Accuracy:", accuracy_score(y_test, y_pred_final))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_final))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_final))

# ==========================================
# 🔟 SAVE MODEL
# ==========================================
best_model.save_model('/content/xgboost_final_tuned.json')

import pickle
with open('/content/feature_order_final.pkl', 'wb') as f:
    pickle.dump(X.columns.tolist(), f)

print("\n✅ FINAL MODEL SAVED!")

Dataset shape: (437711, 26)
scale_pos_weight: 5.479040232096865

Running Hyperparameter Tuning...

Fitting 3 folds for each of 30 candidates, totalling 90 fits

⭐ BEST PARAMETERS:
{'subsample': 0.9, 'reg_lambda': 1.5, 'reg_alpha': 1, 'n_estimators': 1200, 'min_child_weight': 3, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}

===== THRESHOLD TUNING =====
Threshold 0.60 -> Accuracy: 0.7808
Threshold 0.65 -> Accuracy: 0.8047
Threshold 0.70 -> Accuracy: 0.8243
Threshold 0.75 -> Accuracy: 0.8405
Threshold 0.80 -> Accuracy: 0.8505

🔥 BEST THRESHOLD: 0.8000000000000002
🔥 BEST ACCURACY: 0.8505077504769085

====== FINAL MODEL PERFORMANCE ======

Accuracy: 0.8505077504769085
ROC-AUC: 0.824736365002199

Classification Report:
              precision    recall  f1-score   support

         0.0       0.88      0.95      0.92     74031
         1.0       0.53      0.29      0.38     13512

    accuracy                           0.85     87543
   macro avg       0.70     